In [7]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from ipm_new.ipm_solve import ipm_solve # type: ignore[reportMissingImports]
from generators.toy_problem_to_json import toy_problem_to_json # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [9]:
# Number of copies of the network in the toy system
N=5

# Whether the production and flow variables have upper bounds
F_UPPER_BOUNDS=True

# Whether the demand constraint is an equality or inequality
DEMAND_INEQUALITY=True

# Capacity / demand parameters
# GAMMA=1.0
GAMMA=1.0
LAMBD=1.0
EPS=0.1
DELTA=0.1

# Lower bounds and upper bound on pressure
LX=LP=LF=0.0
UP=0.5

LIN_APPROX = 1

# Whether to do an inner or outer approximation, THIS SHOULD STAY TRUE HERE
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [10]:
toy_problem_to_json(N, F_UPPER_BOUNDS, DEMAND_INEQUALITY, GAMMA, LAMBD,
                        EPS, DELTA, LP, LF, UP)
quad, q_time, q_x = solve_qcp_return_x(f_name="toy.json")
print(f"Exact objective value: {quad}")
print(f"Exact solution: {q_x}")

Exact objective value: 5.550002460027112
Exact solution: [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.25127125247977655, 0.25122828552538357, 0.2511799949895359, 0.2511656764680673, 0.2508061129999223, 2.871395207815868e-08, 2.8707678078142255e-08, 2.8770742725580986e-08, 2.8677911604439093e-08, 2.8909715558214754e-08, 0.4987225727179248, 0.007106188137478107, 0.5987319596462213, 0.5012680430005118, 0.4987691531469576, 0.002154129585256837, 0.5987742049877952, 0.5012257976181891, 0.004560717640957947, 0.4988177349829429, 0.0005318771628986975, 0.5988220886941403, 0.5011779139214548, 0.004258202759778427, 0.49883184573587847, 0.0010026614818849974, 0.5988357720360477, 0.5011642306006073, 0.002429901033173434, 0.49919296628647125, 0.0005121445856500206, 0.5991947141728727, 0.5008052883423876, 0.0011001775725685893]


In [11]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                                    f_name="toy.json", endpoints=False)
val, time, x, compl = ipm_solve(f_name="linear_approx.json")
print(x)
print(condition_number_nes_basic())

print(f"Error: {val - quad}")

The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 58 iterations in 1.44 seconds.

Primal objective:   -5.50894149e+00
Dual objective:     -5.50894821e+00

Primal residual:    7.28e-10
Dual residual:      1.69e-09
Complementarity:    6.98e-06

[4.99999430e-01 4.96977189e-01 3.67699294e-01 4.99999437e-01 4.95580162e-01 3.67588879e-01 4.99999438e-01 4.93914545e-01 3.67549922e-01 4.99999440e-01 4.91468555e-01 3.67511567e-01 4.99999452e-01
 4.84565129e-01 3.67346588e-01 3.75000248e-01 3.75000129e-01 3.75000079e-01 3.75000052e-01 3.75000079e-01 5.58249087e-07 5.56899215e-07 5.53416918e-07 5.50222545e-07 5.46214356e-07 3.74998603e-01
 3.45027443e-01 3.92894513e-01 7.07105536e-01 3.74998733e-01 3.41094963e-01 3.92894672e-01 7.07105378e-01 1.29205098e-02 3.74998786e-01 3.36296210e-01 3.92894730e-01 7.07105319e-01 2.19088272e-02
 3.74998815e-01 3.29186327e-01 3.92894755e-01 7.07105294e-01 2.60985042e-02 3.74998813e-01 3.08928187e-01 3.92894

In [12]:
# Define a set of functions for each

quads = [[val] for val in x[(5 * N):]]
# print(quads)
iters = 2

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
                refinement_points = len(qs)
                uniform = np.linspace(lower, upper, num - refinement_points)
                return np.append(uniform, qs)
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="toy.json")
    # quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION,
    #                                             remove_division=REMOVE_DIVISION, f_name="toy.json", endpoints=False)

    val, time, x, compl = ipm_solve(f_name="linear_approx.json", verbose=True)

    # print(f"x: {x}")
    print(f"val: {val}")

    for j, qs in enumerate(quads):
        qs.append(x[(5 * N) + j])

    # print(quads)

print(x)
print(f"Error: {val - quad}")

Iteration 1
11.92462120245875
omega was limited by the max cost
Condition number before pre-conditioning: 8.823904808621858
Condition number after pre-conditioning: 9.239691608593132
Iteration 1:
Primal objective:   3.59265360e+02 
Dual objective:     -1.10337063e+02

Primal residual:    9.21e+01
Dual residual:      1.23e+02
Complementarity:    1.40e+04

Condition number before pre-conditioning: 8.07492926240792
Condition number after pre-conditioning: 8.812320487279328
Iteration 2:
Primal objective:   3.61249026e+02 
Dual objective:     -1.04737994e+02

Primal residual:    8.72e+01
Dual residual:      1.17e+02
Complementarity:    1.33e+04

Condition number before pre-conditioning: 8.10932682600305
Condition number after pre-conditioning: 8.871132148614882
Iteration 3:
Primal objective:   3.62860002e+02 
Dual objective:     -1.00514357e+02

Primal residual:    8.35e+01
Dual residual:      1.12e+02
Complementarity:    1.28e+04

Condition number before pre-conditioning: 8.139380820753736